# Random Lattice Generators Are Not Bad

This notebook is a Julia counterpart of QMCPy's `lattice_random_generator.ipynb`. It keeps the original progression from lattice construction to integration, while replacing the heavier plotting sections with compact printed summaries that are easier to rerun in `QMC.jl`.


Original QMCPy demo: [`QMCPy/demos/lattice_random_generator.ipynb`](../../QMCPy/demos/lattice_random_generator.ipynb)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QMCSoftware/QMC.jl/blob/develop/demos/lattice.ipynb)


In [1]:
using QMC
using Statistics
using Printf

## Lattice declaration and the `gen_samples` function


In [2]:
println("="^60)
println("Basic Lattice Point Generation")
println("="^60)

dd = Lattice(2; randomize=false, seed=7)
x = gen_samples(dd, 8)
println("  Unshifted lattice, 8 points in 2D:")
for i in 1:8
    @printf("    %d: [%.4f  %.4f]\n", i, x[i,1], x[i,2])
end
println()

Basic Lattice Point Generation
  Unshifted lattice, 8 points in 2D:


    1: [0.0000  0.0000]
    2: [0.5000  0.5000]


    3: [0.2500  0.7500]
    4: [0.7500  0.2500]
    5: [0.1250  0.3750]
    6: [0.6250  0.8750]
    7: [0.3750  0.1250]
    8: [0.8750  0.6250]



### Driver code

We start with a small two-dimensional lattice and inspect the generated points directly.


In [3]:
dd_s = Lattice(2; randomize=true, seed=7)
xs = gen_samples(dd_s, 8)
println("  Shifted lattice, 8 points in 2D:")
for i in 1:8
    @printf("    %d: [%.4f  %.4f]\n", i, xs[i,1], xs[i,2])
end
println()

  Shifted lattice, 8 points in 2D:
    1: [0.7177  0.2410]
    2: [0.2177  0.7410]
    3: [0.9677  0.9910]
    4: [0.4677  0.4910]
    5: [0.8427  0.6160]
    6: [0.3427  0.1160]
    7: [0.0927  0.3660]
    8: [0.5927  0.8660]



### Different orderings and randomized shifts

The QMCPy notebook uses scatter plots to show these patterns. Here we print representative samples instead so the notebook remains lightweight and diff-friendly.


In [4]:
println("="^60)
println("Lattice Orderings")
println("="^60)

for order in ["natural", "linear", "radical_inverse", "gray"]
    dd_o = Lattice(2; randomize=false, seed=1, order=order)
    xo = gen_samples(dd_o, 8)
    println("  Order = \"$order\":")
    for i in 1:4
        @printf("    %d: [%.4f  %.4f]\n", i, xo[i,1], xo[i,2])
    end
    println("    ...")
    println()
end

Lattice Orderings
  Order = "natural":
    1: [0.0000  0.0000]
    2: [0.5000  0.5000]
    3: [0.2500  0.7500]
    4: [0.7500  0.2500]
    ...

  Order = "linear":
    1: [0.0000  0.0000]
    2: [0.1250  0.3750]
    3: [0.2500  0.7500]
    4: [0.3750  0.1250]
    ...

  Order = "radical_inverse":
    1: [0.0000  0.0000]
    2: [0.5000  0.5000]
    3: [0.2500  0.7500]
    4: [0.7500  0.2500]
    ...

  Order = "gray":
    1: [0.0000  0.0000]
    2: [0.5000  0.5000]
    3: [0.7500  0.2500]
    4: [0.2500  0.7500]
    ...



## Replications

Independent random shifts produce replications that are useful for uncertainty assessment and diagnostic summaries.


In [5]:
println("="^60)
println("Replications (Independent Random Shifts)")
println("="^60)

dd_r = Lattice(3; randomize=true, seed=42, replications=4)
xr = gen_samples(dd_r, 16)
println("  Shape: $(size(xr))  (R × n × d)")
println()

for r in 1:2
    println("  Replication $r (first 4 points):")
    for i in 1:4
        @printf("    [%.4f  %.4f  %.4f]\n", xr[r,i,1], xr[r,i,2], xr[r,i,3])
    end
    println()
end

Replications (Independent Random Shifts)
  Shape: (4, 16, 3)  (R × n × d)

  Replication 1 (first 4 points):
    [0.3761  0.6972  0.8614]


    [0.8761  0.1972  0.3614]
    [0.6261  0.4472  0.6114]
    [0.1261  0.9472  0.1114]

  Replication 2 (first 4 points):
    [0.8025  0.7346  0.5879]
    [0.3025  0.2346  0.0879]
    [0.0525  0.4846  0.3379]
    [0.5525  0.9846  0.8379]



### Replications for variance estimation

The original notebook continues with mean-versus-median comparisons across repeated randomized lattices. This Julia version keeps a smaller textual summary of the same replication idea.


In [6]:
println("  Mean per replication (dim 1):")
for r in 1:4
    @printf("    Rep %d: mean = %.4f\n", r, mean(xr[r, :, 1]))
end
println()

  Mean per replication (dim 1):
    Rep 1: mean = 0.4698
    Rep 2: mean = 0.5212


    Rep 3: mean = 0.5137
    Rep 4: mean = 0.4818



## High-dimensional lattice

QMC.jl uses the same `qmctoolscl`-backed lattice generator family as QMCPy, so the high-dimensional examples are directly comparable.


In [7]:
println("="^60)
println("High-Dimensional Lattice (Kuo Generating Vector)")
println("="^60)

for d in [10, 100, 1000]
    local xhd
    dd_hd = Lattice(d; randomize=true, seed=7)
    xhd = gen_samples(dd_hd, 2^10)
    @printf("  d = %4d: mean(dim 1) = %.4f, mean(dim %d) = %.4f\n",
            d, mean(xhd[:, 1]), d, mean(xhd[:, d]))
end
println()

High-Dimensional Lattice (Kuo Generating Vector)
  d =   10: mean(dim 1) = 0.4999, mean(dim 10) = 0.5001
  d =  100: mean(dim 1) = 0.4996, mean(dim 100) = 0.5003
  d = 1000: mean(dim 1) = 0.5004, mean(dim 1000) = 0.5000



## Integration

As in the QMCPy notebook, we close with a compact integration example to show how the lattice construction feeds into a full QMC solve.


In [8]:
println("="^60)
println("Integration: Keister function, d=3")
println("="^60)

dd = Lattice(3; randomize=true, seed=7)
tm = Gaussian(dd; covariance=0.5)
f = Keister(tm)
sc = CubQMCLatticeG(f; abs_tol=0.01, n_init=2^8, n_reps=16)
result = integrate(sc)
exact = keister_exact(3)
@printf("  Solution: %.6f (exact = %.6f, error = %.2e)\n",
        result.solution, exact, abs(result.solution - exact))

println()
println("="^60)
println("Lattice demo completed!")

Integration: Keister function, d=3
  Solution: 2.167908 (exact = 2.168309, error = 4.01e-04)



Lattice demo completed!
